# Semantic Search for CS Papers — Colab Training
Run each cell in order. All outputs are saved to Google Drive so they persist across sessions.

## 1. Mount Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Clone Repo & Install Dependencies

In [5]:
import os

REPO_DIR = '/content/semantic-search'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Algebra101/Semantic-Search-For-Computer-Science-papers.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
    print('Repo already cloned, pulled latest changes.')

Cloning into '/content/semantic-search'...
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 27 (delta 3), reused 26 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (27/27), 18.81 KiB | 377.00 KiB/s, done.
Resolving deltas: 100% (3/3), done.


In [6]:
!pip install -q -r /content/semantic-search/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 71.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 133.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 136.1 MB/s eta 0:00:0000:01


## 3. Set Up Paths & Environment Variables

In [7]:
import sys
sys.path.insert(0, '/content/semantic-search')

# All outputs go to Drive so they survive session restarts
BASE_DIR = '/content/drive/MyDrive/SemanticSearch'

os.makedirs(f'{BASE_DIR}/data/raw', exist_ok=True)
os.makedirs(f'{BASE_DIR}/data/processed', exist_ok=True)
os.makedirs(f'{BASE_DIR}/models', exist_ok=True)

os.environ['RAW_ARXIV_FILE']  = f'{BASE_DIR}/data/raw/arxiv-metadata-oai-snapshot.json'
os.environ['DATA_FILE']       = f'{BASE_DIR}/data/processed/projects.csv'
os.environ['TOP_K']           = '10'
os.environ['SEMANTIC_WEIGHT'] = '0.6'
os.environ['LEXICAL_WEIGHT']  = '0.4'
os.environ['VECTOR_SIZE']     = '100'
os.environ['WINDOW']          = '5'
os.environ['MIN_COUNT']       = '1'
os.environ['EPOCHS']          = '20'
os.environ['MAX_RECORDS']     = '1000000'

print('Paths configured:')
print(f"  Raw data : {os.environ['RAW_ARXIV_FILE']}")
print(f"  Data file: {os.environ['DATA_FILE']}")

Paths configured:
  Raw data : /content/drive/MyDrive/SemanticSearch/data/raw/arxiv-metadata-oai-snapshot.json
  Data file: /content/drive/MyDrive/SemanticSearch/data/processed/projects.csv


## 4. Upload arXiv JSON to Drive
**Skip this cell if the file is already in your Drive.**  
Upload `arxiv-metadata-oai-snapshot.json` (5GB) — this may take a while.

In [ ]:
raw_file = os.environ['RAW_ARXIV_FILE']

if os.path.exists(raw_file):
    print(f'File already exists: {raw_file} — skipping upload.')
else:
    from google.colab import files
    import shutil
    print('Select arxiv-metadata-oai-snapshot.json to upload...')
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    shutil.move(fname, raw_file)
    print(f'Moved to: {raw_file}')

Select arxiv-metadata-oai-snapshot.json to upload...


: 

## 5. Prepare arXiv Dataset
Filters CS papers and writes to Drive. **Skip if `projects.csv` already exists in Drive.**

In [ ]:
data_file = os.environ['DATA_FILE']

if os.path.exists(data_file):
    print(f'Dataset already prepared: {data_file} — skipping.')
else:
    %cd /content/semantic-search
    !python src/prepare_arxiv_dataset.py

## 6. Verify Dataset

In [8]:
import pandas as pd

df = pd.read_csv(os.environ['DATA_FILE'], low_memory=False)
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

Shape: (892992, 5)
Columns: ['doc_id', 'title', 'abstract', 'keywords', 'category']


,doc_id,title,abstract,keywords,category
0,704.0002,Sparsity-certifying Graph Decompositions,"We describe a new algorithm, the $(k,\ell)$-...",math.CO cs.CG,math.CO cs.CG
1,704.0046,A limit relation for entropy and channel capac...,"In a quantum mechanical model, Diosi, Feldma...",quant-ph cs.IT math.IT,quant-ph cs.IT math.IT
2,704.0047,Intelligent location of simultaneously active ...,The intelligent acoustic emission locator is...,cs.NE cs.AI,cs.NE cs.AI


## 7. Run Training Pipeline

In [11]:
%cd /content/semantic-search
!python main.py

/content/semantic-search
Loading and preprocessing dataset...
^C


## 8. Verify Outputs

In [10]:
BASE_DIR = '/content/drive/MyDrive/SemanticSearch'

for folder in ['data/processed', 'models']:
    path = f'{BASE_DIR}/{folder}'
    print(f'\n{path}:')
    for f in os.listdir(path):
        size = os.path.getsize(f'{path}/{f}') / 1e6
        print(f'  {f}  ({size:.1f} MB)')


/content/drive/MyDrive/SemanticSearch/data/processed:
  projects.csv  (1163.6 MB)

/content/drive/MyDrive/SemanticSearch/models:
